Step 0 — Pick a job worth queueing. You need a task that's genuinely slow, so the payoff is visible. Good options that build on previous challenges: "generate a summary report of all notes" (reuses your API's data), "import a large CSV" (reuses the Tier 1 CLI logic), or simply a time.sleep(10) fake "model training" job — which foreshadows Tier 4 nicely. The teaching setup: first implement it synchronously as a normal endpoint and feel the 10-second hang. The pain motivates the architecture.

Step 1 — Add Redis and a worker to your compose file. Your Docker challenge pays off immediately — the infrastructure is four lines:

In [ ]:
redis:
image: redis:7
worker:
build: .
command: celery -A tasks worker --loglevel=info
environment:
DATABASE_URL: postgresql://app:secret@db:5432/notes
REDIS_URL: redis://redis:6379/0
depends_on: [redis, db]

Notice the worker builds from the same image as the API — same code, different command. That's the core mental model: a queue system is your app running twice, with Redis as the mailbox between them. Celery is the standard pick and what job postings mention; beginners who find Celery's magic off-putting can use RQ instead, which is delightfully simple and teaches the same concepts.

Step 2 — Write the task and the enqueue endpoint. The minimal loop:

In [ ]:
from celery import Celery

celery_app = Celery("tasks", broker=os.environ["REDIS_URL"],
                    backend=os.environ["REDIS_URL"])

@celery_app.task
def generate_report(user_id: int) -> str:
    time.sleep(10)
    return f"report for {user_id} done"

And in FastAPI, the endpoint pattern that matters: POST /reports calls generate_report.delay(user_id), and returns 202 Accepted with the task ID — not 200 with a result, because there is no result yet. That status code is the whole philosophy of async work in one number. Then add GET /reports/{task_id} that checks AsyncResult(task_id).state and returns PENDING, SUCCESS (with the result), or FAILURE. Watch it work: fire the POST, poll the GET, and keep docker compose logs worker -f open in a second terminal to see the task get picked up. The instant-response POST after feeling the 10-second synchronous version is the "aha" moment.

Step 3 — Break it on purpose. Three experiments that teach more than any tutorial. Kill the worker (docker compose stop worker), enqueue three jobs, restart the worker — watch it drain the backlog; the queue is the buffer. Enqueue ten jobs and watch them run serially, then restart the worker with --concurrency=4 and watch them parallelize. Finally, make the task raise an exception and observe what happens by default: the job just... fails, silently, forever. That uncomfortable discovery is the doorway to the stretch.


Stretch part 1 — Retries done right. The naive fix is @task(autoretry_for=(Exception,), max_retries=3), but the real lesson is in the details: add retry_backoff=True so retries wait 1s, 2s, 4s instead of hammering a struggling database, and be deliberate about which exceptions retry. A network timeout deserves a retry; a ValidationError will fail identically all five times — retrying it just burns cycles. Simulate a flaky dependency with if random.random() < 0.7: raise ConnectionError and watch the backoff pattern in the worker logs.

Stretch part 2 — Idempotency keys. Now the question that separates junior from senior: your task retried — did it run its side effects twice? If the job sends an email or charges a card, retries just became a bug. The fix: the client (or enqueue endpoint) supplies an idempotency key, and the task checks it before doing work — SET key value NX EX 3600 in Redis returns false if the key already exists, making "check and claim" a single atomic operation. Demo it by enqueueing the same job twice with the same key: one runs, one no-ops. This concept resurfaces in payment APIs, Kafka consumers, and ML pipeline steps — it's arguably the single most transferable idea in the whole challenge.

Stretch part 3 — Dead-letter queue. What happens to a job that fails all its retries? By default it evaporates. Build the safety net: use Celery's on_failure handler (or a task_failure signal) to write the task name, arguments, and exception into a failed_jobs Postgres table — your DLQ. Then add two small API endpoints: GET /admin/failed-jobs to inspect the graveyard and POST /admin/failed-jobs/{id}/retry to re-enqueue one. That tiny admin surface is a genuinely production-grade pattern, and it makes a great demo-day moment: kill a job five times, show it in the DLQ, fix the bug, replay it, watch it succeed.